# Biomarker discovery pipeline (ICI first-line)

Runs the `pipelines.biomarkers.*` stages end to end with the active **Python kernel**, each as
a subprocess so no state leaks between stages. Runs **after** [05_run_full_cohort_risk_scores.ipynb](05_run_full_cohort_risk_scores.ipynb)
and **before** [06b_generate_figure_data.ipynb](06b_generate_figure_data.ipynb) — `figures.prep.figure5`
consumes this pipeline's propensity predictions and compiled hits.

### Data source
All inputs come from the PROFILE_DATA parquets. Lines of therapy, line start dates, and ICI
exposure are derived in `pipelines.biomarkers.profile_lines` from `MEDICATIONS_SUMMARY.parquet`
(28-day regimen window, curated checkpoint-inhibitor drug list); somatic markers come from
`GENOMIC_SPECIMEN` + `SOMATIC_WIDE_BY_SAMPLE` anchored at each patient's **line landmark**, not
at first treatment. Stage 0 (`audit_line_derivation`, off by default) is a read-only concordance
check of that derivation against the lab-owned `ALL_MEDICATION_LINES.csv` this pipeline used to
read; run it once after a PROFILE release before trusting the rest.

### Cohorts (`build_line_matched_cohort`)
Both cohorts are restricted to patients with a usable somatic profile at their line-1 landmark
(a non-RAPIDHEME genomic specimen reported on or before that date) — an unsequenced patient has
no marker to test. The restriction is applied before matching so cohort 2's pairs stay intact.

- **cohort1** (`first_line_unmatched`) — all patients observed at line 1; exposure is whether line 1
  contains an ICI. No matching.
- **cohort2** (`first_line_matched`) — same first-line new-user population, 1:1 exact matched on
  cancer type without replacement.

Neither cohort conditions control eligibility on eventual ICI receipt or on the maximum line a
patient later reaches — those post-landmark definitions caused immortal-time/selection leakage in
the earlier design.

### Propensity-score models (trained within each cohort)
- **covariates_only** — CV logistic regression on demographics + cancer type (+ line dummies for cohort 2)
- **covariates_plus_embeddings** — the same covariates plus pooled clinical-text embeddings

Panel version is deliberately excluded from the PS model (it may mediate ICI assignment) and enters
the downstream Cox models as a confounder instead.

### Analysis model (`run_IPTW_analysis`)
Full cohort, predictive interaction:
`S(t) ~ base_vars + line_dummies + marker + PX_on_ICI + marker x ICI`,
fit `ATE` (IPTW) and `noIPTW`. The interaction coefficient is the estimand — does
the marker modify ICI benefit?

An earlier ICI-only prognostic screen ("Track 1") was removed; `track`/`track2_*`
names are retained in the schema and filenames so existing outputs still read.

FDR is applied within mutation type (`_SNV`, `_SV`, `_FUSION`, `_DEL`, `_AMP`) per cancer type.

### If stage 5 finishes but the results are empty
Open [05c_diagnose_iptw_run.ipynb](05c_diagnose_iptw_run.ipynb). It counts rows (not bytes) in
everything `IPTW_runs_*/` holds and rebuilds stage 5's covariate list from the saved IPTW datasets
to name whatever emptied the model frames — no refit required.

In [ ]:
from __future__ import annotations

import os
import re
import subprocess
import sys
import time
from pathlib import Path


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find v2 root from {start}")


V2_ROOT = find_v2_root()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

import config

BIOMARKER_PATH = config.BIOMARKER_PATH
MATCHED_COHORT_PATH = config.MATCHED_COHORT_PATH
COMPILED_DIR = os.path.join(BIOMARKER_PATH, "compiled_results/")

# Grids are module-level constants in the stage scripts; mirrored here for the
# completion checks and summaries below. Change them in the scripts, not here.
COHORTS = ["cohort1", "cohort2"]
PS_MODELS = ["covariates_only", "covariates_plus_embeddings"]
BUFFER = 30
TRACK2_WEIGHTS = ["ATE", "noIPTW"]

# --- Stage toggles: turn off stages you do not need to re-execute ---
RUN_AUDIT = False     # 0: audit_line_derivation (read-only concordance check
                      #    of the PROFILE-derived lines vs MED_LINES_FILE)
RUN_COHORTS = True    # 1: build_line_matched_cohort
RUN_EMBED = True      # 2: ICI_generate_embeddings
RUN_PS = True         # 3: ICI_train_propensity
RUN_IPTW = True       # 4: generate_IPTW_df
RUN_COX = True        # 5: run_IPTW_analysis   <-- the long one
RUN_COMPILE = True    # 6: compile_IPTW_results
RUN_VALIDATE = False  # 7: validate_and_report (optional, local; writes the .docx)

# When True, a stage whose outputs already exist is skipped even if its toggle is on.
SKIP_IF_DONE = True

# Parallelism for stage 5. run_IPTW_analysis resolves its worker count as
# IPTW_N_JOBS -> SLURM_CPUS_PER_TASK -> min(visible cores, 16), and caps every
# worker to one thread (POLARS_MAX_THREADS/OMP_NUM_THREADS=1) so n_jobs workers
# cannot each build a machine-sized Rayon pool and exhaust the cgroup pid limit.
# There is no CLI flag. None => leave whatever is already in the environment.
N_JOBS: int | None = None

# --- Smoke test for stage 5: cap the number of markers screened ---
# A full screen is ~1500 markers x 2 tracks x 2 weightings x cancer types and
# runs for hours. Set MAX_MARKERS to a small number for a fast end-to-end check
# that the stage actually works -- singular design matrices, missing deps, and
# unwritable paths all surface in the first minutes. None => screen everything.
#
# A capped run writes to IPTW_runs_{spec}_smoke/ instead of the real directory,
# so it can never overwrite results and stage 6 will not compile it. Its numbers
# are NOT valid: FDR is computed within mutation type over whatever is screened,
# so a subset changes every q-value. Use it to test plumbing, not to preview hits.
#
# Note SKIP_IF_DONE checks the REAL directory, so a previous full run makes
# stage 5 skip. Set SKIP_IF_DONE = False when smoke testing.
MAX_MARKERS: int | None = None   # e.g. 20 for a quick check

STAGE_ENV = os.environ.copy()
# ICI_train_propensity calls plt.show(); force a non-interactive backend so the
# subprocess cannot block or fail on a headless node. The ROC PNG is still written.
STAGE_ENV["MPLBACKEND"] = "Agg"
# Be explicit so validate_and_report never falls back to its hardcoded legacy
# compiled-results directory (which can silently read a stale result set).
STAGE_ENV["COMPILED_DIR"] = COMPILED_DIR.rstrip("/")
if N_JOBS is not None:
    # IPTW_N_JOBS takes precedence over SLURM_CPUS_PER_TASK, so setting it here
    # wins even inside an allocation that requested a different core count.
    STAGE_ENV["IPTW_N_JOBS"] = str(N_JOBS)
if MAX_MARKERS is not None:
    STAGE_ENV["IPTW_MAX_MARKERS"] = str(MAX_MARKERS)
else:
    # A stale value in the inherited environment would silently cap a real run.
    STAGE_ENV.pop("IPTW_MAX_MARKERS", None)
    STAGE_ENV.pop("IPTW_MARKER_FRACTION", None)

print(f"v2 root:   {V2_ROOT}")
print(f"Python:    {sys.executable}")
print(f"Data root: {config.DATA_PATH}")
print(f"Biomarker: {BIOMARKER_PATH}")
print(f"Compiled:  {COMPILED_DIR}")
print(f"n_jobs (stage 5): {STAGE_ENV.get('IPTW_N_JOBS') or STAGE_ENV.get('SLURM_CPUS_PER_TASK') or 'auto (min(cores, 16))'}")
if MAX_MARKERS is not None:
    print(f"\n*** SMOKE TEST: stage 5 capped at {MAX_MARKERS} markers. "
          f"Writes to IPTW_runs_*_smoke/; results are NOT valid. ***")
    if SKIP_IF_DONE:
        print("    SKIP_IF_DONE is True and checks the real output dir — "
              "set it False or stage 5 may skip.")

## Preconditions

Upstream inputs each stage assumes. A missing file here is the difference between "stage 1 failed"
and "stage 1 never had its inputs" — worth knowing before a long run starts.

In [ ]:
NOTES_META = os.path.join(config.NOTES_PATH, "full_clinical_notes_embeddings_metadata.parquet")
NOTES_ARRAY = os.path.join(config.NOTES_PATH, "full_clinical_notes_embeddings_as_array.npy.zst")

PROFILE = config.PROFILE_DATA_PATH

PRECONDITIONS = [
    ("survival cohort",      os.path.join(config.SURV_PATH, "death_met_surv_df.parquet"), "1,2,3,4"),
    ("full cohort",          os.path.join(config.SURV_PATH, "cohort_df.parquet"),         "1,4"),
    ("PROFILE medications",  os.path.join(PROFILE, "MEDICATIONS_SUMMARY.parquet"),        "1"),
    ("PROFILE cancer types", os.path.join(config.CANCER_ANNOTATIONS_PATH, "CANCER_TYPE.parquet"), "1,4"),
    ("PROFILE specimens",    os.path.join(PROFILE, "GENOMIC_SPECIMEN.parquet"),           "4"),
    ("PROFILE somatic wide", os.path.join(PROFILE, "SOMATIC_WIDE_BY_SAMPLE.parquet"),     "4"),
    ("note embeddings meta", NOTES_META,  "2"),
    ("note embeddings array", NOTES_ARRAY, "2"),
    ("medication lines (audit only)", config.MED_LINES_FILE, "0"),
]

missing = []
for label, path, stages in PRECONDITIONS:
    ok = os.path.exists(path)
    if not ok:
        missing.append(label)
    print(f"[{'ok ' if ok else 'MISSING'}] {label:<22} (stages {stages:<8}) {path}")

if missing:
    print(f"\n{len(missing)} input(s) missing: {', '.join(missing)}")
    print("Stages depending on them will fail. This cell does not raise — inspect and decide.")
else:
    print("\nAll inputs present.")

## Run

Each stage is `python -m pipelines.biomarkers.<module>` with `cwd` set to `v2/`. Output streams
straight through — stage 5 runs for a long time behind per-cancer-type progress, so silence would
be indistinguishable from a hang. A failure stops the pipeline: later stages read earlier stages'
files, so there is nothing useful to continue with.

In [ ]:
def _all_exist(paths: list[str]) -> bool:
    return bool(paths) and all(os.path.exists(p) for p in paths)


def _cohort_files() -> list[str]:
    return [os.path.join(MATCHED_COHORT_PATH, f"matched_cohort_{c}.parquet") for c in COHORTS]


def _embed_files() -> list[str]:
    return [
        os.path.join(config.DATA_PATH, f"treatment_prediction/{c}/prediction_data/",
                     f"w_{BUFFER}_day_buffer/ICI_prediction_df_w_{BUFFER}_day_buffer.parquet")
        for c in COHORTS
    ]


def _ps_files() -> list[str]:
    return [
        os.path.join(config.DATA_PATH, f"treatment_prediction/{c}/",
                     f"{p}_propensity/w_{BUFFER}_day_buffer/predictions.parquet")
        for c in COHORTS for p in PS_MODELS
    ]


def _iptw_files() -> list[str]:
    return [os.path.join(BIOMARKER_PATH, f"IPTW_df_{c}_{p}.parquet")
            for c in COHORTS for p in PS_MODELS]


def _run_dir(cohort: str, ps_model: str) -> str:
    # Always the real directory. A smoke run (MAX_MARKERS) writes to a _smoke
    # suffixed one, so completion checks here deliberately ignore it: a capped
    # run must never satisfy SKIP_IF_DONE for a real one.
    return os.path.join(BIOMARKER_PATH, f"IPTW_runs_{cohort}_{ps_model}/")


def _smoke_run_dir(cohort: str, ps_model: str) -> str:
    return os.path.join(BIOMARKER_PATH, f"IPTW_runs_{cohort}_{ps_model}_smoke/")


def _cox_done() -> bool:
    """Every spec dir has at least one Track 2 ATE result."""
    for cohort in COHORTS:
        for ps_model in PS_MODELS:
            run_path = _run_dir(cohort, ps_model)
            if not os.path.isdir(run_path):
                return False
            if not any(f.endswith("_results.parquet") for f in os.listdir(run_path)):
                return False
    return True


def _compile_files() -> list[str]:
    return [os.path.join(COMPILED_DIR, "track2_all_significant_hits.csv")]


STAGES = [
    # Stage 0 is a read-only report, not a pipeline stage — it writes nothing,
    # so it has no completion marker and re-runs whenever its toggle is on.
    ("0: Line-derivation audit", "audit_line_derivation",  RUN_AUDIT,   lambda: False),
    ("1: Cohort construction", "build_line_matched_cohort", RUN_COHORTS, lambda: _all_exist(_cohort_files())),
    ("2: Embedding datasets",  "ICI_generate_embeddings",   RUN_EMBED,   lambda: _all_exist(_embed_files())),
    ("3: Propensity scores",   "ICI_train_propensity",      RUN_PS,      lambda: _all_exist(_ps_files())),
    ("4: IPTW datasets",       "generate_IPTW_df",          RUN_IPTW,    lambda: _all_exist(_iptw_files())),
    ("5: Cox marker screens",  "run_IPTW_analysis",         RUN_COX,     _cox_done),
    ("6: Compile results",     "compile_IPTW_results",      RUN_COMPILE, lambda: _all_exist(_compile_files())),
    # Stage 7 has no stable completion marker — it re-validates in place, so it always re-runs.
    ("7: Validate + report",   "validate_and_report",       RUN_VALIDATE, lambda: False),
]

timings: list[tuple[str, str, float]] = []
failed: tuple[str, str, int] | None = None

for label, module, enabled, is_done in STAGES:
    if not enabled:
        print(f"\n=== SKIPPED (toggle off)      Stage {label} ===", flush=True)
        timings.append((label, "skipped (toggle)", 0.0))
        continue
    if SKIP_IF_DONE and is_done():
        print(f"\n=== SKIPPED (already complete) Stage {label} ===", flush=True)
        timings.append((label, "skipped (complete)", 0.0))
        continue

    print(f"\n{'=' * 70}\n=== Stage {label}: pipelines.biomarkers.{module}\n{'=' * 70}", flush=True)
    t0 = time.time()
    proc = subprocess.run([sys.executable, "-m", f"pipelines.biomarkers.{module}"],
                          cwd=V2_ROOT, env=STAGE_ENV)
    elapsed = time.time() - t0

    if proc.returncode != 0:
        timings.append((label, f"FAILED (exit {proc.returncode})", elapsed))
        failed = (label, module, proc.returncode)
        print(f"\n[FAIL] Stage {label} ({module}) exited {proc.returncode} after {elapsed / 60:.1f} min.")
        print("Downstream stages not run — they read this stage's output files.")
        break
    timings.append((label, "ok", elapsed))
    print(f"\n[ok] Stage {label} finished in {elapsed / 60:.1f} min.", flush=True)

print(f"\n{'=' * 70}\nStage timings\n{'=' * 70}")
for label, status, elapsed in timings:
    dur = f"{elapsed / 60:8.1f} min" if elapsed else " " * 12
    print(f"  {label:<26} {status:<22} {dur}")
if failed:
    print(f"\nPipeline stopped at stage {failed[0]}.")
else:
    print("\nPipeline complete.")

## Output inventory

What is actually on disk now, stage by stage — so a partial or interrupted run stays legible.

In [ ]:
def _report(label: str, paths: list[str]) -> None:
    found = [p for p in paths if os.path.exists(p)]
    mark = "ok " if len(found) == len(paths) else "PARTIAL" if found else "none"
    print(f"[{mark:>7}] {label:<24} {len(found)}/{len(paths)}")
    for p in paths:
        if p not in found:
            print(f"            missing: {p}")


print(f"Data root: {config.DATA_PATH}\n")
_report("1 matched cohorts", _cohort_files())
_report("2 embedding datasets", _embed_files())
_report("3 PS predictions", _ps_files())
_report("4 IPTW datasets", _iptw_files())

smoke_dirs = [(c, p) for c in COHORTS for p in PS_MODELS
              if os.path.isdir(_smoke_run_dir(c, p))]
if smoke_dirs:
    print(f"\n[note] {len(smoke_dirs)} smoke-test run director(ies) present "
          f"(IPTW_runs_*_smoke/). Capped marker subsets — not valid results, "
          f"not compiled by stage 6.")

print("\n5 Cox marker screens:")
cancer_types: set[str] = set()
for cohort in COHORTS:
    for ps_model in PS_MODELS:
        run_path = _run_dir(cohort, ps_model)
        spec = f"{cohort}_{ps_model}"
        if not os.path.isdir(run_path):
            print(f"  [none] {spec:<42} (no directory)")
            continue
        names = os.listdir(run_path)
        types_here = {m.group(1) for m in (re.match(r"(.+)_results\.parquet$", n) for n in names) if m}
        cancer_types |= types_here
        n_res = sum(1 for n in names if n.endswith("_results.parquet"))
        n_diag = sum(1 for n in names if n.endswith("_diagnostics.parquet"))
        print(f"  [ok  ] {spec:<42} {len(types_here):>3} cancer types, "
              f"{n_res:>3} results + {n_diag:>3} diagnostics files")

if cancer_types:
    print(f"\n  Cancer types screened ({len(cancer_types)}): {', '.join(sorted(cancer_types))}")

print()
_report("6 compiled hits", _compile_files())
_report("7 validation + report", [os.path.join(COMPILED_DIR, "all_findings_with_validation.csv"),
                                  os.path.join(COMPILED_DIR, "ICI_Biomarker_Pipeline_Report.docx")])

## Results summary

FDR-significant hits, cross-specification consistency, and propensity diagnostics. Every read is
guarded, so this section is safe to run on a partial pipeline.

In [ ]:
import polars as pl

TOP_N = 15

(t2_path,) = _compile_files()
t2 = pl.read_csv(t2_path) if os.path.exists(t2_path) else None

if t2 is None:
    print("Compiled hits not found — run stage 6 (compile_IPTW_results) first.")
else:
    for name, df, p_col, hr_col in [
        ("Marker x ICI interaction", t2, "p_markerxICI", "HR_markerxICI"),
    ]:
        print(f"\n{'=' * 70}\n{name}: {df.height} FDR-significant hits\n{'=' * 70}")
        if df.is_empty():
            continue
        with pl.Config(tbl_rows=-1, tbl_cols=-1):
            print(df.group_by(["cohort", "ps_model", "weight_type"])
                    .agg(pl.len().alias("n_hits"),
                         pl.col("cancer_type").n_unique().alias("n_cancer_types"))
                    .sort(["cohort", "ps_model", "weight_type"]))
            cols = [c for c in ["marker", "cancer_type", "cohort", "ps_model", "weight_type",
                                hr_col, p_col, "extreme_hr_flag"] if c in df.columns]
            print(f"\nTop {TOP_N} by p-value:")
            print(df.select(cols).sort(p_col).head(TOP_N))

In [ ]:
# Cross-specification consistency for Track 2 — the "robust hit" criterion
# figures.prep.figure5 uses for the manuscript panel. Agreement here is a quick
# sanity check on that figure.
if t2 is not None and not t2.is_empty():
    robust = (t2
              .with_columns(pl.concat_str(["cohort", "ps_model", "weight_type"], separator="_").alias("spec"))
              .group_by(["marker", "cancer_type"])
              .agg(pl.col("spec").n_unique().alias("n_specs"),
                   pl.col("HR_markerxICI").mean().alias("mean_HR"),
                   pl.col("p_markerxICI").min().alias("min_p"),
                   ((pl.col("HR_markerxICI") > 1).all() | (pl.col("HR_markerxICI") < 1).all())
                   .alias("direction_consistent"),
                   pl.col("spec").sort().str.join(", ").alias("specs"))
              .filter(pl.col("n_specs") > 1)
              .sort(["n_specs", "min_p"], descending=[True, False]))
    print(f"Markers significant in >1 specification: {robust.height}")
    with pl.Config(tbl_rows=40, fmt_str_lengths=120):
        print(robust)
else:
    print("No Track 2 hits to cross-check.")

In [ ]:
# Propensity diagnostics: PS AUC and effective sample size per specification.
diag_path = os.path.join(COMPILED_DIR, "scheme_diagnostics_summary.parquet")
if os.path.exists(diag_path):
    diag = pl.read_parquet(diag_path).with_columns(
        ((pl.col("ESS_ATE_treated") + pl.col("ESS_ATE_control"))
         / (pl.col("N_treated") + pl.col("N_control"))).alias("ESS_frac")
    )
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        print(diag.select(["cohort", "ps_model", "cancer_type", "N_treated", "N_control",
                           "PS_AUC", "ESS_ATE_treated", "ESS_ATE_control", "ESS_frac"])
                  .sort("ESS_frac"))
    weak = diag.filter(pl.col("ESS_frac") < 0.5)
    if not weak.is_empty():
        print(f"\n{weak.height} specification(s) retain <50% of their sample as effective size — "
              "weight instability, treat those estimates cautiously.")
else:
    print("Diagnostics summary not found — run stage 6 first.")

In [ ]:
# Propensity ROC curves written by stage 3 (the subprocess runs headless, so the
# figure is only on disk).
from IPython.display import Image, display

for cohort in COHORTS:
    png = os.path.join(config.DATA_PATH, f"treatment_prediction/{cohort}/propensity_ROC_{cohort}.png")
    if os.path.exists(png):
        print(f"{cohort}: {png}")
        display(Image(filename=png))
    else:
        print(f"{cohort}: no ROC figure — run stage 3 (ICI_train_propensity) first.")

## Where everything lands

Paths relative to `DATA_PATH` (see [`../config.py`](../config.py)).

| Stage | Module | Writes |
|---|---|---|
| 1 | `build_line_matched_cohort` | `biomarker_analysis/matched_cohorts/matched_cohort_{cohort1,cohort2}.parquet` |
| 2 | `ICI_generate_embeddings` | `treatment_prediction/{cohort}/prediction_data/w_30_day_buffer/ICI_prediction_df_w_30_day_buffer.parquet`, `.../prediction_times.parquet` |
| 3 | `ICI_train_propensity` | `treatment_prediction/{cohort}/{ps_model}_propensity/w_30_day_buffer/predictions.parquet`, `treatment_prediction/{cohort}/propensity_ROC_{cohort}.png` |
| 4 | `generate_IPTW_df` | `biomarker_analysis/IPTW_df_{cohort}_{ps_model}.parquet` |
| 5 | `run_IPTW_analysis` | `biomarker_analysis/IPTW_runs_{cohort}_{ps_model}/{cancer_type}_results.parquet` and `{cancer_type}_diagnostics.parquet` |
| 6 | `compile_IPTW_results` | `biomarker_analysis/compiled_results/track{1,2}_all_significant_hits.csv`, `cohort_patient_counts.parquet`, `scheme_diagnostics_summary.parquet` |
| 7 | `validate_and_report` | `biomarker_analysis/compiled_results/all_findings_with_validation.csv`, `ICI_Biomarker_Pipeline_Report.docx` |

**Next:** [06b_generate_figure_data.ipynb](06b_generate_figure_data.ipynb). Its `figures.prep.figure5`
reads the stage-3 propensity predictions (panel A) and the stage-6 compiled hits (panels B/C/E).